# 🏥 Tech Challenge FIAP - Fase 1
## Notebook 02: Pré-processamento de Dados

---

### 📋 **Objetivos deste Notebook**

1. ✅ Carregar dados do Notebook 01
2. ✅ Separar features e target
3. ✅ Dividir em conjunto de treino e teste
4. ✅ Criar pipeline de pré-processamento
5. ✅ Normalizar/Padronizar features
6. ✅ Salvar dados processados para modelagem

---

**Input**: Dataset completo da EDA
**Output**: Dados prontos para modelagem (X_train, X_test, y_train, y_test)

## 📚 1. Importação de Bibliotecas

In [ ]:
# Manipulação de dados
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-processamento
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline

# Salvamento
import joblib
import pickle

# Configurações
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✅ Bibliotecas importadas com sucesso!")

## 📂 2. Carregamento dos Dados

In [ ]:
# Carregar dados processados do Notebook 01
try:
    df = pd.read_parquet('../data/processed/breast_cancer_completo.parquet')
    print("✅ Dataset carregado do Parquet")
except:
    df = pd.read_csv('../data/processed/breast_cancer_completo.csv')
    print("✅ Dataset carregado do CSV")

print(f"📊 Dimensões: {df.shape}")
print(f"📋 Colunas: {df.shape[1]}")
print(f"📏 Linhas: {df.shape[0]:,}")

In [ ]:
# Verificar primeiras linhas
print("📊 Primeiras 5 linhas:")
df.head()

In [ ]:
# Informações do dataset
print("ℹ️ INFORMAÇÕES DO DATASET")
print("="*80)
df.info()

## 🎯 3. Separação de Features e Target

In [ ]:
# Separar features (X) e target (y)
# Remover colunas não numéricas e target
X = df.drop(['target', 'diagnosis'], axis=1, errors='ignore')
y = df['target']

print("✅ Features e Target separados")
print(f"\n📊 Features (X):")
print(f"   Shape: {X.shape}")
print(f"   Colunas: {X.shape[1]}")
print(f"   Tipo: {X.dtypes.value_counts().to_dict()}")

print(f"\n🎯 Target (y):")
print(f"   Shape: {y.shape}")
print(f"   Valores únicos: {y.nunique()}")
print(f"   Distribuição:\n{y.value_counts()}")
print(f"   Proporção:\n{y.value_counts(normalize=True) * 100}")

In [ ]:
# Verificar se há valores ausentes
print("❓ VERIFICAÇÃO DE VALORES AUSENTES")
print("="*80)

missing_X = X.isnull().sum().sum()
missing_y = y.isnull().sum()

print(f"Features (X): {missing_X} valores ausentes")
print(f"Target (y): {missing_y} valores ausentes")

if missing_X == 0 and missing_y == 0:
    print("\n✅ EXCELENTE! Nenhum valor ausente.")
else:
    print("\n⚠️ Atenção: Valores ausentes detectados!")

## 🔀 4. Divisão em Treino e Teste

In [ ]:
# Definir seed para reprodutibilidade
RANDOM_STATE = 42
TEST_SIZE = 0.3  # 70% treino, 30% teste

# Divisão estratificada (mantém proporção das classes)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y  # Mantém proporção de classes
)

print("✅ Dados divididos em treino e teste")
print("="*80)
print(f"\n📊 CONJUNTO DE TREINO:")
print(f"   X_train: {X_train.shape}")
print(f"   y_train: {y_train.shape}")
print(f"   Proporção: {len(X_train)/len(X)*100:.1f}%")

print(f"\n📊 CONJUNTO DE TESTE:")
print(f"   X_test: {X_test.shape}")
print(f"   y_test: {y_test.shape}")
print(f"   Proporção: {len(X_test)/len(X)*100:.1f}%")

print(f"\n🎯 RANDOM_STATE: {RANDOM_STATE} (para reprodutibilidade)")

In [ ]:
# Verificar balanceamento das classes após split
print("⚖️ VERIFICAÇÃO DE BALANCEAMENTO")
print("="*80)

print("\n📊 Distribuição no conjunto de TREINO:")
train_dist = y_train.value_counts(normalize=True) * 100
print(train_dist)

print("\n📊 Distribuição no conjunto de TESTE:")
test_dist = y_test.value_counts(normalize=True) * 100
print(test_dist)

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Treino
y_train.value_counts().plot(kind='bar', ax=axes[0], color=['#FF6B6B', '#4ECDC4'], alpha=0.8)
axes[0].set_title('Distribuição - Conjunto de Treino', fontweight='bold')
axes[0].set_xlabel('Classe')
axes[0].set_ylabel('Contagem')
axes[0].set_xticklabels(['Maligno (0)', 'Benigno (1)'], rotation=0)
for i, v in enumerate(y_train.value_counts().values):
    axes[0].text(i, v, f'{v}\n({train_dist.values[i]:.1f}%)', ha='center', va='bottom')

# Teste
y_test.value_counts().plot(kind='bar', ax=axes[1], color=['#FF6B6B', '#4ECDC4'], alpha=0.8)
axes[1].set_title('Distribuição - Conjunto de Teste', fontweight='bold')
axes[1].set_xlabel('Classe')
axes[1].set_ylabel('Contagem')
axes[1].set_xticklabels(['Maligno (0)', 'Benigno (1)'], rotation=0)
for i, v in enumerate(y_test.value_counts().values):
    axes[1].text(i, v, f'{v}\n({test_dist.values[i]:.1f}%)', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('../reports/figures/08_train_test_split.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Gráfico salvo em: reports/figures/08_train_test_split.png")

## 🔧 5. Pipeline de Pré-processamento

### **Por que usar StandardScaler?**

- Features têm escalas diferentes (ex: radius 0-30, area 0-2500)
- Muitos algoritmos (SVM, KNN, Regressão Logística) são sensíveis à escala
- StandardScaler transforma para média=0 e desvio padrão=1

**Fórmula**: `z = (x - μ) / σ`

In [ ]:
# Criar pipeline de pré-processamento
preprocessing_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

print("✅ Pipeline criado:")
print(preprocessing_pipeline)

In [ ]:
# Treinar o pipeline APENAS com dados de treino
# IMPORTANTE: Nunca usar dados de teste no fit!
print("🔧 Treinando pipeline com dados de TREINO...")
X_train_scaled = preprocessing_pipeline.fit_transform(X_train)
print("✅ Pipeline treinado")

# Transformar dados de teste (sem fit!)
print("\n🔧 Transformando dados de TESTE...")
X_test_scaled = preprocessing_pipeline.transform(X_test)
print("✅ Dados de teste transformados")

print(f"\n📊 Shapes após transformação:")
print(f"   X_train_scaled: {X_train_scaled.shape}")
print(f"   X_test_scaled: {X_test_scaled.shape}")

## 📊 6. Verificação da Normalização

In [ ]:
# Estatísticas ANTES da normalização
print("📊 ANTES DA NORMALIZAÇÃO (Primeiras 5 features)")
print("="*80)
print("\nTREINO:")
print(X_train.iloc[:, :5].describe())

print("\n" + "="*80)
print("📊 DEPOIS DA NORMALIZAÇÃO (Primeiras 5 features)")
print("="*80)
print("\nTREINO:")
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
print(X_train_scaled_df.iloc[:, :5].describe())

print("\n✅ Observe que após StandardScaler:")
print("   • Média ≈ 0")
print("   • Desvio padrão ≈ 1")

In [ ]:
# Visualização: Antes vs Depois da normalização
# Selecionar 4 features para visualizar
features_to_plot = X.columns[:4]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# ANTES da normalização
for idx, col in enumerate(features_to_plot):
    axes[0, idx].hist(X_train[col], bins=30, alpha=0.7, color='steelblue', edgecolor='black')
    axes[0, idx].set_title(f'ANTES: {col}', fontsize=10, fontweight='bold')
    axes[0, idx].set_xlabel('Valor')
    axes[0, idx].set_ylabel('Frequência')
    axes[0, idx].grid(alpha=0.3)

# DEPOIS da normalização
for idx, col in enumerate(features_to_plot):
    col_idx = X.columns.get_loc(col)
    axes[1, idx].hist(X_train_scaled[:, col_idx], bins=30, alpha=0.7, color='coral', edgecolor='black')
    axes[1, idx].set_title(f'DEPOIS: {col}', fontsize=10, fontweight='bold')
    axes[1, idx].set_xlabel('Valor Normalizado')
    axes[1, idx].set_ylabel('Frequência')
    axes[1, idx].axvline(0, color='red', linestyle='--', linewidth=2, label='Média = 0')
    axes[1, idx].legend(fontsize=8)
    axes[1, idx].grid(alpha=0.3)

plt.suptitle('Efeito da Normalização (StandardScaler)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/09_normalizacao_efeito.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico salvo em: reports/figures/09_normalizacao_efeito.png")

## 💾 7. Salvamento dos Dados Processados

In [ ]:
# Criar diretório se não existir
import os
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

print("📁 Diretórios verificados/criados")

In [ ]:
# Salvar arrays NumPy (mais eficiente para ML)
np.save('../data/processed/X_train.npy', X_train_scaled)
np.save('../data/processed/X_test.npy', X_test_scaled)
np.save('../data/processed/y_train.npy', y_train)
np.save('../data/processed/y_test.npy', y_test)

print("✅ Arrays salvos em formato NumPy (.npy)")

In [ ]:
# Salvar também nomes das features
feature_names = X.columns.tolist()
with open('../data/processed/feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

print("✅ Nomes das features salvos")
print(f"📋 Total de features: {len(feature_names)}")

In [ ]:
# Salvar o pipeline treinado (IMPORTANTE!)
joblib.dump(preprocessing_pipeline, '../models/preprocessing_pipeline.joblib')
print("✅ Pipeline de pré-processamento salvo em: models/preprocessing_pipeline.joblib")

print("\n💡 Este pipeline será usado para transformar novos dados!")

In [ ]:
# Salvar informações do split
split_info = {
    'test_size': TEST_SIZE,
    'random_state': RANDOM_STATE,
    'n_samples_train': len(X_train),
    'n_samples_test': len(X_test),
    'n_features': X_train.shape[1],
    'feature_names': feature_names,
    'class_distribution_train': y_train.value_counts().to_dict(),
    'class_distribution_test': y_test.value_counts().to_dict()
}

with open('../data/processed/split_info.pkl', 'wb') as f:
    pickle.dump(split_info, f)

print("✅ Informações do split salvas")

## 📊 8. Resumo do Pré-processamento

In [ ]:
print("="*80)
print("📊 RESUMO DO PRÉ-PROCESSAMENTO")
print("="*80)

print("\n✅ ETAPAS REALIZADAS:")
print("   1. ✅ Carregamento dos dados")
print("   2. ✅ Separação de features e target")
print("   3. ✅ Train-test split (70/30)")
print("   4. ✅ Normalização com StandardScaler")
print("   5. ✅ Dados salvos")

print("\n📊 DIMENSÕES DOS DADOS:")
print(f"   • Total de amostras: {len(X):,}")
print(f"   • Features: {X.shape[1]}")
print(f"   • Treino: {X_train_scaled.shape}")
print(f"   • Teste: {X_test_scaled.shape}")

print("\n🎯 DISTRIBUIÇÃO DAS CLASSES:")
print("   TREINO:")
for cls, count in y_train.value_counts().items():
    pct = count / len(y_train) * 100
    print(f"      Classe {cls}: {count} ({pct:.2f}%)")
print("   TESTE:")
for cls, count in y_test.value_counts().items():
    pct = count / len(y_test) * 100
    print(f"      Classe {cls}: {count} ({pct:.2f}%)")

print("\n💾 ARQUIVOS SALVOS:")
print("   • data/processed/X_train.npy")
print("   • data/processed/X_test.npy")
print("   • data/processed/y_train.npy")
print("   • data/processed/y_test.npy")
print("   • data/processed/feature_names.pkl")
print("   • data/processed/split_info.pkl")
print("   • models/preprocessing_pipeline.joblib")

print("\n🔧 TRANSFORMAÇÕES APLICADAS:")
print("   • StandardScaler (z-score normalization)")
print("   • Média ≈ 0, Desvio Padrão ≈ 1")

print("\n⚙️ PARÂMETROS:")
print(f"   • Random State: {RANDOM_STATE}")
print(f"   • Test Size: {TEST_SIZE} ({TEST_SIZE*100:.0f}%)")
print(f"   • Stratify: Sim (mantém proporção de classes)")

print("\n" + "="*80)

## 📝 9. Próximos Passos

### **✅ Concluído neste Notebook:**
- Dados carregados e verificados
- Features e target separados
- Train-test split realizado (70/30)
- Normalização aplicada (StandardScaler)
- Dados salvos em formato eficiente
- Pipeline salvo para reutilização

### **➡️ Próximo Notebook: `03_modelagem.ipynb`**

No próximo notebook, vamos:
1. Carregar dados pré-processados
2. Treinar múltiplos modelos de classificação:
   - Regressão Logística
   - Random Forest
   - Support Vector Machine (SVM)
   - Gradient Boosting (XGBoost)
3. Salvar modelos treinados
4. Fazer predições iniciais

### **🎯 Pontos Importantes:**

1. **Reprodutibilidade**:
   - Random state fixado (42)
   - Pipeline salvo
   - Parâmetros documentados

2. **Boas Práticas**:
   - Pipeline treinado APENAS com dados de treino
   - Estratificação mantém proporção de classes
   - Normalização apropriada para algoritmos sensíveis à escala

3. **Considerações Médicas**:
   - Dados balanceados (bom para classificação)
   - Sem data leakage (teste isolado)
   - Pipeline replicável para novos pacientes

---

**⚠️ LEMBRETE IMPORTANTE:**
> Este pré-processamento garante que nossos modelos terão a melhor chance  
> de aprender padrões reais, sem vieses de escala ou vazamento de dados.

---

## ✅ Conclusão do Notebook 02

**Status**: ✅ Completo

**Tempo de Execução**: [Anotar após execução]

**Próximo**: `03_modelagem.ipynb`